# Trend pattern map -- scatter exploration

Loads roughly 1000 trend rows from [`trending.sql`](trending.sql) and
renders scatter-style charts that make clusters, outliers, and regimes easy to spot.

**Setup (once):** this notebook uses the same isolated environment as the other notebooks.

```bash
cd notebooks
uv sync
```

Connection settings are read from the repository root `.env`
(`POSTGRES_HOST`, `POSTGRES_USER`, `POSTGRES_DATABASE`, `POSTGRES_PASSWORD`,
`PGSSLMODE`). For Azure set `PGSSLMODE=require`.

In [ ]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import psycopg
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")

conninfo = psycopg.conninfo.make_conninfo(
    host=os.environ["POSTGRES_HOST"],
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    dbname=os.environ["POSTGRES_DATABASE"],
    sslmode=os.getenv("PGSSLMODE", "require"),
)
print("Connecting to", os.environ["POSTGRES_HOST"])

In [ ]:
target_rows = 1500

sql_path = Path("trending.sql")
sql = sql_path.read_text()

# Keep query logic in sync with trending.sql, but increase output size for pattern work.
sql = re.sub(r"LIMIT\s+\d+\s*;?\s*$", f"LIMIT {target_rows};", sql, flags=re.IGNORECASE | re.MULTILINE)

with psycopg.connect(conninfo) as conn:
    df = pd.read_sql(sql, conn)

print(f"Loaded {len(df)} rows")
df.head(20)

In [ ]:
# Derived columns make scale differences easier to see in one view.
plot_df = df.copy()
plot_df["log_recent_count"] = np.log10(plot_df["recent_count"] + 1)
plot_df["log_growth_ratio"] = np.log10(plot_df["growth_ratio"] + 1)

plot_df[["words", "recent_count", "growth_ratio", "z_score", "recent_sources"]].head()

## Main pattern scatter\n
\n
X = current volume, Y = growth ratio, point size = number of corroborating sources,\n
color = statistical surprise (z-score).\n

In [ ]:
fig = px.scatter(
    plot_df,
    x="recent_count",
    y="growth_ratio",
    size="recent_sources",
    color="z_score",
    color_continuous_scale="Turbo",
    hover_name="words",
    hover_data=["baseline_count", "absolute_growth", "n_gram"],
    title="Pattern map: growth vs. recent volume (about 1000 points)",
)
fig.update_layout(height=680, xaxis_title="recent occurrences", yaxis_title="growth ratio")
fig.show()

## Log-scale scatter (better cluster separation)\n
\n
Compresses very large values so medium and small clusters become visible.\n

In [ ]:
fig = px.scatter(
    plot_df,
    x="log_recent_count",
    y="log_growth_ratio",
    size="recent_sources",
    color="z_score",
    color_continuous_scale="Inferno",
    hover_name="words",
    hover_data=["recent_count", "growth_ratio", "baseline_count", "n_gram"],
    title="Pattern map (log10 axes): cluster structure",
)
fig.update_layout(
    height=680,
    xaxis_title="log10(recent_count + 1)",
    yaxis_title="log10(growth_ratio + 1)",
)
fig.show()

## Scatter matrix (multi-dimensional pattern scan)\n
\n
Each panel is a scatter plot of two metrics, which helps spot non-linear relationships\n
and unusual combinations quickly.\n

In [ ]:
matrix_df = plot_df[[
    "words",
    "recent_count",
    "baseline_count",
    "growth_ratio",
    "z_score",
    "recent_sources",
]].copy()

fig = px.scatter_matrix(
    matrix_df,
    dimensions=["recent_count", "baseline_count", "growth_ratio", "z_score", "recent_sources"],
    color="z_score",
    color_continuous_scale="Viridis",
    hover_name="words",
    title="Scatter matrix of trend signals",
)
fig.update_layout(height=900)
fig.show()

In [ ]:
# Quick outlier list to pair with the visuals.
plot_df.sort_values(["z_score", "recent_sources"], ascending=False)[
    ["words", "recent_count", "baseline_count", "growth_ratio", "z_score", "recent_sources"]
].head(30)

In [ ]:
import re

noise_patterns = [
    r"\b(parser|navbox|hlist|mw)\b",
    r"^\d+(\s+\d+){1,}$",
    r"\b\d+\b",
    r"\b(low|high|none)\b",
]

def is_noise_phrase(text: str) -> bool:
    t = str(text).strip().lower()
    if any(re.search(p, t) for p in noise_patterns):
        return True
    # reject mostly non-letter tokens
    letters = sum(ch.isalpha() for ch in t)
    return letters < max(2, int(0.4 * len(t)))

clean = df[~df["words"].apply(is_noise_phrase)].copy()
clean = clean[(clean["recent_sources"] >= 5) & (clean["recent_count"] >= 20)].copy()

print(f"Clean candidates: {len(clean)} / {len(df)}")

interesting = clean.sort_values(
    ["recent_sources", "z_score", "recent_count"], ascending=[False, False, False]
 )[ [
    "words", "recent_count", "recent_sources", "baseline_count", "growth_ratio", "z_score"
 ] ]

interesting.head(40)

In [ ]:
# Second-pass triage: stronger quality filter + simple topic buckets
import re

topic_rules = {
    "security": r"\b(cve|vulnerability|exploit|api key|api keys|breach|malware|patch)\b",
    "ai": r"\b(ai|llm|model|kimi|openai|anthropic|gpt)\b",
    "fediverse": r"\b(mastodon|lemmy|fediverse)\b",
    "dev-howto": r"\b(git clone|pip install|quick start|license mit)\b",
}

generic_noise = r"\b(and|or|to|of|the|other hand|wide range|years and|more people|s and|software and|language and)\b"

c2 = clean.copy()
c2 = c2[~c2["words"].str.lower().str.contains(generic_noise, regex=True, na=False)]

def topic_for_phrase(w: str) -> str:
    t = str(w).lower()
    for topic, pat in topic_rules.items():
        if re.search(pat, t):
            return topic
    return "other"

c2["topic"] = c2["words"].apply(topic_for_phrase)

print("Rows with baseline_count > 0:", int((df["baseline_count"] > 0).sum()))
print("Rows after strict filtering:", len(c2))
print(c2["topic"].value_counts().to_string())

c2.sort_values(["recent_sources", "recent_count"], ascending=False)[
    ["topic", "words", "recent_count", "recent_sources", "growth_ratio", "z_score"]
] .head(60)

In [ ]:
diag_sql = """
SELECT
  min("timestamp") AS min_ts,
  max("timestamp") AS max_ts,
  COUNT(*) FILTER (WHERE "timestamp" > (SELECT max("timestamp") FROM public.ngrams) - INTERVAL '7 days') AS recent_rows,
  COUNT(*) FILTER (WHERE "timestamp" <= (SELECT max("timestamp") FROM public.ngrams) - INTERVAL '7 days'
                   AND "timestamp" > (SELECT max("timestamp") FROM public.ngrams) - INTERVAL '37 days') AS baseline_rows
FROM public.ngrams;
"""

with psycopg.connect(conninfo) as conn:
    diag = pd.read_sql(diag_sql, conn)

diag

In [ ]:
# True rising trends: require baseline evidence (not brand-new phrases)
sql_existing = """
WITH params AS (
  SELECT
    (SELECT max("timestamp") FROM public.ngrams) AS ref_ts,
    INTERVAL '7 days' AS recent_window,
    INTERVAL '30 days' AS baseline_window
),
recent AS (
  SELECT words, n_gram, COUNT(*) AS recent_count, COUNT(DISTINCT source) AS recent_sources
  FROM public.ngrams, params
  WHERE "timestamp" > ref_ts - recent_window
  GROUP BY words, n_gram
),
baseline AS (
  SELECT words, n_gram, COUNT(*) AS baseline_count
  FROM public.ngrams, params
  WHERE "timestamp" <= ref_ts - recent_window
    AND "timestamp" > ref_ts - recent_window - baseline_window
  GROUP BY words, n_gram
),
joined AS (
  SELECT
    r.words, r.n_gram, r.recent_count, r.recent_sources, b.baseline_count,
    r.recent_count::numeric / 7.0 AS recent_rate,
    b.baseline_count::numeric / 30.0 AS baseline_rate
  FROM recent r
  JOIN baseline b USING (words, n_gram)
  WHERE r.n_gram >= 2
    AND r.recent_count >= 8
    AND b.baseline_count >= 5
    AND r.recent_sources >= 5
 )
SELECT
  words, n_gram, recent_count, recent_sources, baseline_count,
  ROUND((recent_rate + 1.0/7)/(baseline_rate + 1.0/30), 2) AS growth_ratio,
  ROUND((recent_count - baseline_rate*7.0)/SQRT(GREATEST(baseline_rate*7.0, 1.0)), 2) AS z_score
FROM joined
ORDER BY growth_ratio DESC, z_score DESC, recent_sources DESC
LIMIT 200;
"""

with psycopg.connect(conninfo) as conn:
    df_existing = pd.read_sql(sql_existing, conn)

# Light cleanup for obvious parser/template artifacts
noise_rx = r"\b(parser|navbox|hlist|reflist|list style|font size|font weight|background color|output div|mw)\b|^\d+(\s+\d+){1,}$"
df_existing_clean = df_existing[~df_existing["words"].str.lower().str.contains(noise_rx, regex=True, na=False)].copy()

print(f"Existing-trend candidates: {len(df_existing)} | after cleanup: {len(df_existing_clean)}")
df_existing_clean.head(40)

In [ ]:
# Final curation: remove encoded artifacts and phrase boilerplate
artifact_rx = r"\b(39|34|gt|lt|mw|parser|output|reflist|navbox|hlist)\b|\b\d+\b"
boilerplate_rx = r"\b(and i|but i|what s|it s|you re|don t|doesn t|that s|u s|e g|a gt|id gt|s and|code and)\b"

curated = df_existing_clean.copy()
curated = curated[~curated["words"].str.lower().str.contains(artifact_rx, regex=True, na=False)]
curated = curated[~curated["words"].str.lower().str.contains(boilerplate_rx, regex=True, na=False)]

print(f"Curated trend rows: {len(curated)}")

top_by_sources = curated.sort_values(["recent_sources", "recent_count"], ascending=False)[
    ["words", "recent_count", "recent_sources", "baseline_count", "growth_ratio", "z_score"]
] .head(25)

top_by_growth = curated.sort_values(["growth_ratio", "recent_sources"], ascending=False)[
    ["words", "recent_count", "recent_sources", "baseline_count", "growth_ratio", "z_score"]
] .head(25)

print("Top by source corroboration")
display(top_by_sources)
print("Top by growth")
display(top_by_growth)